# smsdk Example 3 — Cookbooks

Listing cookbooks, inspecting their structure, pulling top runs for a recipe group, and reading current tag values.

For full method signatures and parameter reference, see [docs/README.md](../docs/README.md#cookbooks).

**Updated:** April 2026 — replaces the former `Cookbooks Examples.ipynb`.

> **Terminology:** what the UI calls **Products** are represented in the SDK as **recipe groups**.

## Setup

In [1]:
from smsdk import client
import pandas as pd
import os

In [ ]:
# Set before running this notebook.
tenant = "demo-bottling"
api_key = ""
api_secret = ""

In [3]:
cli = client.Client(tenant)
success = cli.login('apikey', key_id=api_key, secret_id=api_secret)
assert success, 'SDK login failed — check tenant / API key / secret.'

## List cookbooks

`get_cookbooks()` returns every cookbook on the tenant — both deployed and undeployed. Each cookbook is a nested dict; the top-level DataFrame below just surfaces name / assets / id.

In [4]:
cookbooks = cli.get_cookbooks()
print(f'Total cookbooks on this tenant: {len(cookbooks)}')

df_cookbooks = pd.DataFrame(cookbooks)
df_cookbooks[['name', 'assetNames', 'id']]

Total cookbooks on this tenant: 21


,name,assetNames,id
0,L1: Performance and Quality,"[Blender_1, Packer_2, Packer_4, Filler_1, L1_W...",66980b5f337841084825c5b9
1,Line One Efficiency,"[L1_FullCan_Conveyor, Blender_1, Palletizer_2,...",669aab34dc43a850b5b995ec
2,DEMO - L1: Performance and Quality,"[Blender_1, Packer_2, Packer_4, Filler_1, L1_W...",675313e872adb6a058ba16b3
3,L1 - Filler #2 (1B): Essence,"[Filler_2, Filler_1, Blender_1, L1_Warmer, Fil...",67e1be02956f86513155e297
4,L1 - Empty Can Conveyor: VFD_102_Fault_2_desc,"[Empty_Can_Conveyor, Filler_1, Filtecs_Combine...",68dd822be4f770ee248500e2
5,DELETE L1 - Filler #1 (1A): Expected CIP Type,[Filler_1],68e40c70477426a04ff5df3e
6,Packer : CIP,"[Packer_2, Packer_3, Packer_4]",68f639b4e7d03b83685af115
7,Filler : CIP,"[Filler_2, Filler_1]",68f92190fd233d005af6849f
8,Filler : 5000 Cases Threshold,"[Filler_1, Filler_2]",68f930e4a81c2e550ed616f0
9,Filler : 3000 Cases Threshold,"[Filler_1, Filler_2]",69206ba9dc54668f74640532


### Settings — pick a cookbook and recipe group (edit me)
Adjust these indices to point at a cookbook + product that exist on your tenant. All subsequent cells use these values.

In [5]:
cookbook_idx = 0          # which cookbook to explore
recipe_group_idx = 0      # which product within that cookbook

cookbook = cookbooks[cookbook_idx]
recipe_group = cookbook['recipe_groups'][recipe_group_idx]

print(f"Cookbook: {cookbook['name']}")
print(f"Product:  {recipe_group.get('values')}")
print(f"Assets:   {cookbook.get('assetNames')}")

Cookbook: L1: Performance and Quality
Product:  ['Cola']
Assets:   ['Blender_1', 'Packer_2', 'Packer_4', 'Filler_1', 'L1_Warmer', 'Packer_3', 'Filtecs_Combined', 'Filler_2']


## Inspect the cookbook structure

Each cookbook has one or more **recipe groups** (products). Each recipe group has **outcomes** (what you're optimizing), **levers** (the knobs you can turn), and **constraints** (conditions that partition the data).

In [6]:
print('Outcomes:')
for o in recipe_group['outcomes']:
    print(f"  {o['field']['fieldName']}  (weight: {o['weight']})")

print('\nLevers:')
for lever in recipe_group['levers']:
    print(f"  {lever['fieldName']}")

print('\nConstraints (Conditions):')
for c in recipe_group['constraints']:
    print(f"  {c['field']['fieldName']}")

Outcomes:
  stats__CPM__val  (weight: 1)
  availability  (weight: 1)

Levers:
  stats__HMI_Auto_Man_Speed_Sel__val
  stats__HMI_Auto_Man_PBLT__val
  stats__HMI_Infeed_Primed_PL__val
  stats__AcDriveInput__val
  stats__B254[19]__val
  stats__Filler_No1_1A_SPEED___val
  stats__Filler_No1_Infeed_Counter__val
  stats__Filler_No1_Discharge_Counter__val
  stats__GPS_1125OUT4__val
  stats__Low_Gas_Purge_PB_LT_Bit__val
  stats__Low_Gas_Purge_PB_Light__val

Constraints (Conditions):


In [7]:
# All recipe-group IDs within the selected cookbook
recipe_group_ids = [(rg['id'], rg.get('values')) for rg in cookbook['recipe_groups']]
recipe_group_ids

[('HkPPotruC', ['Cola']),
 ('HyxPwjtHuR', ['Cola Zero Sugar']),
 ('rJbPPiFB_0', [' V Coke']),
 ('HyzDwsFBd0', ['STRAWBERRY']),
 ('HymPPjYBuA', ['Pepper Zero Sugar']),
 ('SyNPvsKHO0', ['Cola Zero SUGAR']),
 ('S1SDPsYrdA', ['Ginger Ale']),
 ('ryUvPjYrOA', ['ROOT BEER']),
 ('BJwvvjKB_C', ['Spiced']),
 ('S1dPDsYH_C', ['Lemon Lime NEW']),
 ('SkKwPoYBOA', ['Dr Easy']),
 ('SJcwPjtSuC', ['Cola ZERO']),
 ('S1swvsYBuC', ['Vintage']),
 ('BynwvsFBu0', ['Diet Cola']),
 ('SJpDvitBu0', ['PEPPER']),
 ('BJAvDiYB_A', ['ORANGE']),
 ('By1ewPoFrdA', ['Zero Sugar Spiced']),
 ('H1xxvwjtHdC', ['Lemon Lime Zero']),
 ('Sy-lwPjFSOC', ['COLA ZERO']),
 ('ByMePPoYHd0', ['DIET Cola'])]

## Top runs for a recipe group

`get_cookbook_top_results(recipe_group_id, limit)` returns the top runs for a product, sorted according to the cookbook's outcome weights. Two views come back:
- **`runs`** — one row per run (matches the UI's Runs view)
- **`constraint_groups`** — one row per recipe (matches the UI's Recipes view)

> The cookbook must be **deployed**; an undeployed cookbook returned by `get_cookbooks()` will error here.

In [8]:
results = cli.get_cookbook_top_results(recipe_group['id'], limit=10)
runs = results['runs']
recipes = results['constraint_groups']

print(f'Runs returned:    {len(runs)}')
print(f'Recipes returned: {len(recipes)}')

Runs returned:    10
Recipes returned: 1


In [9]:
# Sample run — inspect structure
runs[0] if runs else 'No runs'

{'_count': 3,
 '_count_muted': 0,
 '_duration_seconds': 80.0,
 '_earliest': '2025-09-08T05:11:40+00:00',
 '_latest': '2025-09-08T05:13:00+00:00',
 '_score': 0.9216773792245491,
 'constraint_group_id': '0',
 'constraints': [],
 'cookbook': '66980b5f337841084825c5b9',
 'i_vals': [{'name': 'group', 'asset': 'SHARED', 'value': '0'},
  {'name': 'sequence', 'asset': 'SHARED', 'value': 17}],
 'filters': [],
 'levers': [{'name': 'stats__HMI_Auto_Man_Speed_Sel__val',
   'asset': 'Filler_1',
   'd_pos': 3,
   'value': {'min': 1.0,
    'max': 1.0,
    'avg': 1.0,
    'var_pop': 0.0,
    'count': 2.0}},
  {'name': 'stats__HMI_Auto_Man_PBLT__val',
   'asset': 'Filler_1',
   'd_pos': 4,
   'value': {'min': 1.0,
    'max': 1.0,
    'avg': 1.0,
    'var_pop': 0.0,
    'count': 2.0}},
  {'name': 'stats__HMI_Infeed_Primed_PL__val',
   'asset': 'Filler_1',
   'd_pos': 5,
   'value': {'min': 1.0,
    'max': 1.0,
    'avg': 1.0,
    'var_pop': 0.0,
    'count': 2.0}},
  {'name': 'stats__AcDriveInput__val',

In [10]:
# Sample recipe (constraint group) — inspect structure
recipes[0] if recipes else 'No recipes'

{'_score': 0.8436440625156457,
 '_count': 900,
 '_count_muted': 0,
 '_run_count': 10,
 '_duration_seconds': 53062.0,
 '_earliest': '2024-05-08T04:44:26+00:00',
 '_latest': '2025-11-06T09:32:00+00:00',
 'constraint_group_id': '0',
 'constraints': [],
 'filters': [],
 'levers': [{'name': 'stats__HMI_Auto_Man_Speed_Sel__val',
   'asset': 'Filler_1',
   'd_pos': 3,
   'value': {'avg': 0.8957169609416799,
    'normal': None,
    'std': 0.30204612131804326,
    'max': 1.0,
    'min': 0.0}},
  {'name': 'stats__HMI_Auto_Man_PBLT__val',
   'asset': 'Filler_1',
   'd_pos': 4,
   'value': {'avg': 0.8957169609416799,
    'normal': None,
    'std': 0.30204612131804326,
    'max': 1.0,
    'min': 0.0}},
  {'name': 'stats__HMI_Infeed_Primed_PL__val',
   'asset': 'Filler_1',
   'd_pos': 5,
   'value': {'avg': 0.8727314071696094,
    'normal': None,
    'std': 0.30607699845632763,
    'max': 1.0,
    'min': 0.0}},
  {'name': 'stats__AcDriveInput__val',
   'asset': 'Filler_1',
   'd_pos': 6,
   'value':

In [11]:
# Quick stats: how many runs have unmuted records?
if runs:
    unmuted = [r for r in runs if r['_count'] > r['_count_muted']]
    print(f'Total runs:       {len(runs)}')
    print(f'With unmuted data: {len(unmuted)}')

Total runs:       10
With unmuted data: 10


In [12]:
# Flat runs table
df_runs = pd.DataFrame(runs)
df_runs.head()

,_count,_count_muted,_duration_seconds,_earliest,_latest,_score,constraint_group_id,constraints,cookbook,i_vals,filters,levers,outcomes
0,3,0,80.0,2025-09-08T05:11:40+00:00,2025-09-08T05:13:00+00:00,0.921677,0,[],66980b5f337841084825c5b9,"[{'name': 'group', 'asset': 'SHARED', 'value':...",[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_..."
1,3,0,73.0,2025-09-08T09:00:47+00:00,2025-09-08T09:02:00+00:00,0.900177,0,[],66980b5f337841084825c5b9,"[{'name': 'group', 'asset': 'SHARED', 'value':...",[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_..."
2,7,0,325.0,2025-09-08T04:44:35+00:00,2025-09-08T04:50:00+00:00,0.895646,0,[],66980b5f337841084825c5b9,"[{'name': 'group', 'asset': 'SHARED', 'value':...",[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_..."
3,7,0,321.0,2025-09-08T07:39:39+00:00,2025-09-08T07:45:00+00:00,0.885946,0,[],66980b5f337841084825c5b9,"[{'name': 'group', 'asset': 'SHARED', 'value':...",[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_..."
4,4,0,152.0,2025-09-08T07:52:28+00:00,2025-09-08T07:55:00+00:00,0.842583,0,[],66980b5f337841084825c5b9,"[{'name': 'group', 'asset': 'SHARED', 'value':...",[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_..."


In [13]:
# Flat recipes table (matches the Recipes view in the UI)
df_recipes = pd.DataFrame(recipes)
df_recipes.head()

,_score,_count,_count_muted,_run_count,_duration_seconds,_earliest,_latest,constraint_group_id,constraints,filters,levers,outcomes,cookbook
0,0.843644,900,0,10,53062.0,2024-05-08T04:44:26+00:00,2025-11-06T09:32:00+00:00,0,[],[],[{'name': 'stats__HMI_Auto_Man_Speed_Sel__val'...,"[{'name': 'stats__CPM__val', 'asset': 'Filler_...",66980b5f337841084825c5b9


## Normalize range-based constraints to strings

`normalize_constraints` turns a list of range-constraint dicts (each with `to` / `from` / inclusivity flags) into compact string labels like `[120,None)`. Useful for printing recipes. Only works on continuous constraints — skip this for categorical ones.

In [14]:
if runs and runs[0].get('constraints'):
    ranges = [c['values'] for c in runs[0]['constraints']]
    print('Raw:')
    for r in ranges:
        print(f'  {r}')
    print('\nNormalized:')
    for s in cli.normalize_constraints(ranges):
        print(f'  {s}')

## Current tag values (`get_cookbook_current_value`)

`get_cookbook_current_value(variables, minutes)` returns the most recent value of one or more tags. Useful for comparing live process state to the top-run recipes above.

- **`variables`**: list of `{'asset': machine_name, 'name': field_name}` dicts. Both must be the **system names**, not display names — copy them off a run returned by `get_cookbook_top_results` if in doubt.
- **`minutes`**: lookback window (default 1440 = 1 day). Returns `None` for any tag with no readings in that window.
- One invalid entry will error the whole call without telling you which one.

In [15]:
# Pull asset + tag names off the first run if available; otherwise edit these by hand.
if runs and runs[0].get('constraints'):
    example_constraint = runs[0]['constraints'][0]
    vars_to_check = [{
        'asset': runs[0].get('asset'),
        'name': example_constraint['field']['fieldName'],
    }]
    print('Querying:', vars_to_check)
    vals = cli.get_cookbook_current_value(vars_to_check)
    display(pd.DataFrame(vals))
else:
    print('No runs/constraints available to derive an example from — edit the cell to hard-code asset/name.')

No runs/constraints available to derive an example from — edit the cell to hard-code asset/name.


In [16]:
# Example: short lookback returns None if no data in the window
if runs and runs[0].get('constraints'):
    vals = cli.get_cookbook_current_value(vars_to_check, minutes=0.5)
    pd.DataFrame(vals)